In [1]:
import os
import yaml
import json
import logging
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import torch
import pandas as pd
import seaborn as sns
from datetime import datetime
from tensorboard.backend.event_processing import event_accumulator

# Stable Baselines3
from stable_baselines3 import PPO
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor, VecFrameStack, DummyVecEnv
from stable_baselines3.common.utils import set_random_seed

# Grafik
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.style.use('seaborn-v0_8-whitegrid')

# Logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Donanım
def get_optimal_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    elif torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_optimal_device()
print(f"🚀 Donanım: {DEVICE}")

🚀 Donanım: cuda


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
# --- 1. Mac/MPS Fix ---
class Float32ObservationWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.observation_space = gym.spaces.Box(
            low=env.observation_space.low,
            high=env.observation_space.high,
            shape=env.observation_space.shape,
            dtype=np.float32
        )
    def observation(self, observation):
        return np.array(observation, dtype=np.float32)

# --- 2. Fizik Motoru (Fault Injection Ready) ---
class RandomDampingWrapper(gym.Wrapper):
    def __init__(self, env, min_damping, max_damping):
        super().__init__(env)
        self.min_d = min_damping
        self.max_d = max_damping

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        new_damping = self.np_random.uniform(self.min_d, self.max_d)
        self.set_damping(new_damping)
        return self.env.reset(seed=seed, options=options)

    def set_damping(self, value):
        if hasattr(self.env.unwrapped, 'model'):
            dof_count = len(self.env.unwrapped.model.dof_damping)
            if isinstance(value, (float, int)):
                val_array = np.full(dof_count, value)
            else:
                val_array = value
            self.env.unwrapped.model.dof_damping[:] = val_array

# --- 3. Augmented Observation (Context Memory) ---
class AugmentedObservationWrapper(gym.Wrapper):
    """
    Obs + Action(t-1) + Reward(t-1) + Done(t-1)
    """
    def __init__(self, env):
        super().__init__(env)
        old_space = env.observation_space
        action_dim = env.action_space.shape[0]
        
        # Yeni Boyut: Obs + Action + Reward(1) + Done(1)
        new_shape = (old_space.shape[0] + action_dim + 2,)
        
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=new_shape, dtype=np.float32)
        
        # Bufferlar
        self.prev_action = np.zeros(action_dim, dtype=np.float32)
        self.prev_reward = 0.0
        self.prev_done = 0.0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.prev_action = np.zeros_like(self.prev_action)
        self.prev_reward = 0.0
        self.prev_done = 0.0
        return self._get_aug_obs(obs), info

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated
        
        aug_obs = self._get_aug_obs(obs)
        
        self.prev_action = action.astype(np.float32)
        self.prev_reward = float(reward)
        self.prev_done = float(done)
        
        return aug_obs, reward, terminated, truncated, info

    def _get_aug_obs(self, current_obs):
        return np.concatenate([
            current_obs, self.prev_action, [self.prev_reward], [self.prev_done]
        ]).astype(np.float32)

In [3]:
class ScientificTrainer:
    def __init__(self, config_path: str):
        # 1. Config Dosyasını Diskten Oku
        if not os.path.exists(config_path):
            raise FileNotFoundError(
                f"❌ HATA: Config dosyası bulunamadı!\n"
                f"Lütfen '{config_path}' dosyasını elle oluşturun."
            )
            
        with open(config_path) as f:
            self.config = yaml.safe_load(f)
            
        # 2. Run ID ve Loglama
        self.timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.run_id = f"{self.config['experiment']['name']}_{self.timestamp}"
        self.device = DEVICE
        
        # Log Klasörü: ../data/logs/[RUN_ID]
        self.log_dir = os.path.join("..", "data", "logs", self.run_id)
        os.makedirs(self.log_dir, exist_ok=True)
        
        # 3. Config Kopyasını Kaydet (Evidence)
        with open(os.path.join(self.log_dir, "config.json"), "w") as f:
            json.dump(self.config, f, indent=4)
            
        self.print_obsidian_header(config_path)

    def print_obsidian_header(self, source_path):
        print("\n" + "="*40)
        print("📋 OBSIDIAN LAB NOTE")
        print("="*40)
        print(f"run_id: \"{self.run_id}\"")
        print(f"source_config: \"{source_path}\"")
        print(f"device: \"{self.device}\"")
        print("="*40 + "\n")

    def _make_env_factory(self, seed, rank, wrapper_conf, force_cpu=False):
        def _init():
            # Config'den ortam ID'sini al (Reacher-v4/v5)
            env = gym.make(self.config['env']['id'])
            
            device_type = "cpu" if force_cpu else str(self.device)
            if device_type == "mps": 
                env = Float32ObservationWrapper(env)
            
            env = RandomDampingWrapper(
                env, 
                min_damping=wrapper_conf['min_damping'], 
                max_damping=wrapper_conf['max_damping']
            )
            env = AugmentedObservationWrapper(env) # Yeni Wrapper
            
            env.reset(seed=seed + rank)
            return env
        return _init

    def train_variant(self, variant_name):
        print(f"\n{'='*60}")
        print(f"🚀 TRAINING: {variant_name}")
        print(f"{'='*60}")
        
        seeds = self.config['training']['seeds']
        wrapper_conf = self.config['env']['wrappers'][0]['args']
        train_timesteps = self.config['training']['total_timesteps']
        
        trained_model_paths = []

        # Model Seçimi
        if variant_name == "LSTM":
            ModelClass = RecurrentPPO
            policy_type = "MlpLstmPolicy"
            current_device = self.device
            force_cpu = False
            policy_kwargs = self.config['hyperparameters']['policy_kwargs']
        else:
            ModelClass = PPO
            policy_type = "MlpPolicy"
            current_device = "cpu" # CPU Optimization for MLP
            force_cpu = True
            # LSTM parametrelerini temizle
            base_kwargs = self.config['hyperparameters']['policy_kwargs'].copy()
            exclude = ['lstm_hidden_size', 'n_lstm_layers', 'shared_lstm', 'enable_critic_lstm']
            policy_kwargs = {k: v for k, v in base_kwargs.items() if k not in exclude}

        for seed in seeds:
            set_random_seed(seed)
            
            env = SubprocVecEnv([
                self._make_env_factory(seed, i, wrapper_conf, force_cpu) 
                for i in range(self.config['env']['n_envs'])
            ])
            
            if variant_name == "FrameStack":
                env = VecFrameStack(env, n_stack=4)
            
            env = VecMonitor(env, filename=os.path.join(self.log_dir, f"{variant_name}_seed_{seed}_monitor.csv"))
            
            model = ModelClass(
                policy=policy_type,
                env=env,
                verbose=1,
                device=current_device,
                tensorboard_log=self.log_dir,
                learning_rate=self.config['hyperparameters']['learning_rate'],
                n_steps=self.config['hyperparameters']['n_steps'],
                batch_size=self.config['hyperparameters']['batch_size'],
                gamma=self.config['hyperparameters']['gamma'],
                gae_lambda=self.config['hyperparameters']['gae_lambda'],
                ent_coef=self.config['hyperparameters']['ent_coef'],
                policy_kwargs=policy_kwargs
            )

            print(f"   🌱 Seed {seed} eğitiliyor... ({train_timesteps} steps)")
            
            # Progress bar KAPALI, Log Interval SIK
            model.learn(
                total_timesteps=train_timesteps, 
                tb_log_name=f"{variant_name}_seed_{seed}",
                progress_bar=False, 
                log_interval=1 
            )
            
            save_path = os.path.join(self.log_dir, f"final_model_{variant_name}_seed_{seed}")
            model.save(save_path)
            trained_model_paths.append(save_path)
            
            env.close()
            print(f"   ✅ Kaydedildi: {save_path}")
            
        return trained_model_paths

In [4]:
# 1. Config Dosyasını Tanımla
config_path = "../configs/phase_3_1/reacher_experiment_final.yaml"

# 2. Trainer'ı Başlat
trainer = ScientificTrainer(config_path)

# 3. Eğitimleri Sırasıyla Başlat
# Vanilla -> FrameStack -> LSTM
paths_vanilla = trainer.train_variant("Vanilla")
paths_framestack = trainer.train_variant("FrameStack")
paths_lstm = trainer.train_variant("LSTM")

print("\n🎉 TÜM EĞİTİMLER TAMAMLANDI.")


📋 OBSIDIAN LAB NOTE
run_id: "Phase-3.1-MetaRL-Augmented_20251125_072434"
source_config: "../configs/phase_3_1/reacher_experiment_final.yaml"
device: "cuda"


🚀 TRAINING: Vanilla


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cpu device
   🌱 Seed 42 eğitiliyor... (500000 steps)
Logging to ../data/logs/Phase-3.1-MetaRL-Augmented_20251125_072434/Vanilla_seed_42_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -61.2    |
| time/              |          |
|    fps             | 7672     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 4096     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -58.9       |
| time/                   |             |
|    fps                  | 3201        |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.008374874 |
|    clip_fraction        | 0.0419      |
|    clip_ra

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -60.1    |
| time/              |          |
|    fps             | 7206     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 4096     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -59         |
| time/                   |             |
|    fps                  | 3108        |
|    iterations           | 2           |
|    time_elapsed         | 2           |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.010218496 |
|    clip_fraction        | 0.0601      |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.79       |
|    explained_variance   | -0.015      |
|    learning_rate        | 0.

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.



Using cuda device
   🌱 Seed 42 eğitiliyor... (500000 steps)
Logging to ../data/logs/Phase-3.1-MetaRL-Augmented_20251125_072434/LSTM_seed_42_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -60.2    |
| time/              |          |
|    fps             | 2419     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 4096     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 50          |
|    ep_rew_mean          | -59.8       |
| time/                   |             |
|    fps                  | 321         |
|    iterations           | 2           |
|    time_elapsed         | 25          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.004332493 |
|    clip_fraction        | 0.0303      |
|    clip_rang